In [1]:
import warnings
warnings.simplefilter(action='ignore')

import numpy as np
import pandas as pd
import random
import os
import gc
import collections
from tqdm.auto import tqdm
from matplotlib import pyplot as plt

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

In [3]:
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [4]:
class GCF:
    INPUT_ROOT = "/kaggle/input/model-data-obd/data"
    SEED = 0
    MAX_LEN = 1024
    EVAL_MAX_LEN = 1024
    N_FOLDS = 5
    
    BS = 256
    HIDDEN_SIZE = 128
    N_EPOCHS = 80 * 1
    
    LR = 1e-3
    WEIGHT_DECAY = 1e-5

def set_seed(seed=GCF.SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

In [5]:
# Load and prepare data
train = pd.read_csv("/kaggle/input/zelestra-phase-2-data/Dataset 1.csv")
train['datetime'] = pd.to_datetime(train['datetime'])

In [6]:
def create_time_features(df):
    """Add temporal features that help model understand day/night cycles"""
    df = df.copy()
    
    # Extract time components
    df['hour'] = df['datetime'].dt.hour
    df['day_of_year'] = df['datetime'].dt.dayofyear
    df['month'] = df['datetime'].dt.month
    
    # Solar-specific features
    df['is_daylight'] = ((df['hour'] >= 6) & (df['hour'] <= 18)).astype(int)
    df['is_peak_sun'] = ((df['hour'] >= 10) & (df['hour'] <= 14)).astype(int)
    
    # Cyclical encoding for time (important for neural networks)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)
    
    return df

# Add time features
train = create_time_features(train)

In [7]:
def is_night_time(dt):
    t = dt.time()
    return t >= pd.to_datetime("20:00").time() or t <= pd.to_datetime("05:30").time()

print("=== PREPROCESSING WITH TEMPORAL AWARENESS ===")
for col, dtype in zip(train.columns, train.dtypes):
    if dtype == 'O' or col == 'datetime' or col == 'ttr_potenciaproducible':
        continue
    if col in ['hour', 'day_of_year', 'month', 'is_daylight', 'is_peak_sun', 
               'hour_sin', 'hour_cos', 'day_sin', 'day_cos']:
        continue  # Skip our engineered features
    
    print(f"Processing {col}: {train[col].isna().sum()} missing values")
    
    # Use ONLY time information for imputation (no target usage)
    night_mask = train['datetime'].map(is_night_time)
    
    # For solar/irradiance features, set to 0 at night
    if any(keyword in col.lower() for keyword in ['gii', 'ghi', 'ir_cel', 'pv_i']):
        train.loc[train[col].isna() & night_mask, col] = 0
    else:
        # For other features (temperature, humidity), use time-aware interpolation
        train.loc[train[col].isna() & night_mask, col] = train[train['is_daylight'] == 0][col].median()
    
    # Fill remaining with overall median
    train[col].fillna(train[col].median(), inplace=True)

=== PREPROCESSING WITH TEMPORAL AWARENESS ===
Processing meteorolgicas_em_03_02_gii: 639 missing values
Processing meteorolgicas_em_08_01_gii: 44 missing values
Processing meteorolgicas_em_03_02_ghi: 634 missing values
Processing meteorolgicas_em_08_01_ghi: 49 missing values
Processing meteorolgicas_em_08_01_gii_rear: 137 missing values
Processing meteorolgicas_em_03_02_gii_rear: 717 missing values
Processing meteorolgicas_em_03_02_desviacin_incidente: 599 missing values
Processing meteorolgicas_em_08_01_desviacin_incidente: 302 missing values
Processing meteorolgicas_em_03_02_t_amb: 593 missing values
Processing meteorolgicas_em_08_01_t_amb: 4 missing values
Processing meteorolgicas_em_03_02_h_r: 593 missing values
Processing meteorolgicas_em_08_01_h_r: 5 missing values
Processing meteorolgicas_em_03_02_t_dlogger: 609 missing values
Processing meteorolgicas_em_08_01_t_dlogger: 10 missing values
Processing meteorolgicas_em_08_01_ws: 10 missing values
Processing meteorolgicas_em_03_02_w

In [8]:
features = ['meteorolgicas_em_03_02_gii',
 'meteorolgicas_em_08_01_gii',
 'meteorolgicas_em_03_02_ghi',
 'meteorolgicas_em_08_01_ghi',
 'meteorolgicas_em_08_01_gii_rear',
 'meteorolgicas_em_03_02_gii_rear',
 'meteorolgicas_em_03_02_t_amb',
 'meteorolgicas_em_08_01_t_amb',
 'meteorolgicas_em_03_02_h_r',
 'meteorolgicas_em_08_01_h_r',
 'meteorolgicas_em_03_02_t_dlogger',
 'meteorolgicas_em_08_01_t_dlogger',
 'celulas_ctin08_cc_08_1_ir_cel_1',
 'celulas_ctin08_cc_08_2_ir_cel_1',
 'celulas_ctin03_cc_03_1_ir_cel_1',
 'celulas_ctin03_cc_03_2_ir_cel_1',
 'celulas_ctin08_cc_08_2_ir_cel_2',
 'celulas_ctin03_cc_03_2_ir_cel_2',
 'celulas_ctin03_cc_03_1_ir_cel_2',
 'celulas_ctin08_cc_08_1_ir_cel_2',
 'celulas_ctin08_cc_08_2_t_amb',
 'celulas_ctin03_cc_03_1_t_amb',
 'celulas_ctin08_cc_08_1_t_amb',
 'celulas_ctin03_cc_03_2_t_amb',
 'celulas_ctin03_cc_03_1_t_mod',
 'celulas_ctin08_cc_08_2_t_mod',
 'celulas_ctin03_cc_03_2_t_mod',
 'celulas_ctin08_cc_08_1_t_mod',
 'inversores_ctin03_strings_string8_pv_i9',
 'inversores_ctin03_strings_string8_pv_i13',
 'inversores_ctin03_strings_string8_pv_i1',
 'inversores_ctin03_strings_string8_pv_i6',
 'inversores_ctin03_strings_string8_pv_i4',
 'inversores_ctin03_strings_string8_pv_i11',
 'inversores_ctin03_strings_string8_pv_i10',
 'inversores_ctin03_strings_string8_pv_i2',
 'inversores_ctin03_strings_string8_pv_i5',
 'inversores_ctin03_strings_string8_pv_i7',
 'inversores_ctin03_strings_string8_pv_i12',
 'inversores_ctin03_strings_string8_pv_i3',
 'inversores_ctin03_strings_string8_pv_i8',
 'inversores_ctin03_strings_string10_pv_i9',
 'inversores_ctin03_strings_string10_pv_i7',
 'inversores_ctin03_strings_string10_pv_i4',
 'inversores_ctin03_strings_string10_pv_i12',
 'inversores_ctin03_strings_string10_pv_i8',
 'inversores_ctin03_strings_string10_pv_i13',
 'inversores_ctin03_strings_string10_pv_i10',
 'inversores_ctin03_strings_string10_pv_i6',
 'inversores_ctin03_strings_string10_pv_i11',
 'inversores_ctin03_strings_string10_pv_i5',
 'inversores_ctin03_strings_string10_pv_i1',
 'inversores_ctin03_strings_string10_pv_i3',
 'inversores_ctin03_strings_string10_pv_i2',
 'inversores_ctin08_strings_string9_pv_i12',
 'inversores_ctin08_strings_string9_pv_i13',
 'inversores_ctin08_strings_string9_pv_i1',
 'inversores_ctin08_strings_string9_pv_i7',
 'inversores_ctin08_strings_string9_pv_i8',
 'inversores_ctin08_strings_string9_pv_i6',
 'inversores_ctin08_strings_string9_pv_i10',
 'inversores_ctin08_strings_string9_pv_i9',
 'inversores_ctin08_strings_string9_pv_i11',
 'inversores_ctin08_strings_string9_pv_i4',
 'inversores_ctin08_strings_string9_pv_i5',
 'inversores_ctin08_strings_string9_pv_i2',
 'inversores_ctin08_strings_string9_pv_i3',
 'inversores_ctin08_strings_string12_pv_i4',
 'inversores_ctin08_strings_string12_pv_i1',
 'inversores_ctin08_strings_string12_pv_i5',
 'inversores_ctin08_strings_string12_pv_i7',
 'inversores_ctin08_strings_string12_pv_i3',
 'inversores_ctin08_strings_string12_pv_i6',
 'inversores_ctin08_strings_string12_pv_i2',
 'inversores_ctin08_strings_string12_pv_i8',
 'inversores_ctin08_strings_string12_pv_i9',
 'inversores_ctin08_strings_string12_pv_i10',
 'inversores_ctin08_inv_08_08_p',
 'inversores_ctin03_inv_03_03_p',
 'inversores_ctin03_inv_03_03_p_dc',
 'inversores_ctin08_inv_08_08_p_dc',
 'inversores_ctin03_inv_03_03_eact_tot',
 'inversores_ctin08_inv_08_08_eact_tot']


In [9]:
time_features = ['hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'is_daylight', 'is_peak_sun']
all_features = features + time_features

In [10]:
train = train[all_features + ['ttr_potenciaproducible']]

In [11]:
train['ttr_potenciaproducible'].fillna(train['ttr_potenciaproducible'].mean(),inplace=True)

In [12]:
# Check for NaN values and infinite values
print("=== DATA QUALITY CHECK ===")
print(f"NaN values in features: {train[all_features].isna().sum().sum()}")
print(f"Infinite values in features: {np.isinf(train[all_features]).sum().sum()}")
print(f"NaN values in target: {train['ttr_potenciaproducible'].isna().sum()}")
print(f"Infinite values in target: {np.isinf(train['ttr_potenciaproducible']).sum()}")

=== DATA QUALITY CHECK ===
NaN values in features: 0
Infinite values in features: 0
NaN values in target: 0
Infinite values in target: 0


In [13]:
train_data, test_data = train_test_split(train, test_size=0.33, random_state=GCF.SEED)

In [14]:
# Scale the features and target
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

In [15]:
# Fit scalers on training data only
train_features_scaled = feature_scaler.fit_transform(train_data[all_features])
train_target_scaled = target_scaler.fit_transform(train_data['ttr_potenciaproducible'].values.reshape(-1, 1)).flatten()

test_features_scaled = feature_scaler.transform(test_data[all_features])
test_target_scaled = target_scaler.transform(test_data['ttr_potenciaproducible'].values.reshape(-1, 1)).flatten()

# Create scaled dataframes
train_scaled = pd.DataFrame(train_features_scaled, columns=all_features)
train_scaled['ttr_potenciaproducible'] = train_target_scaled

test_scaled = pd.DataFrame(test_features_scaled, columns=all_features)
test_scaled['ttr_potenciaproducible'] = test_target_scaled


In [16]:
LR = 1e-3
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
max_length = 50
HIDDEN_SIZE = 256
N_LAYERS = 2
N_CLF_LAYERS = 2
BIDIRECTIONAL = True
DROP = 0.3
EPOCHS = 50

In [17]:
class ZelestraData(Dataset):
    def __init__(self, df, target='ttr_potenciaproducible', max_length=None):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.target = target
        self.max_length = max_length
        self.features = [col for col in df.columns if col != self.target]
        
        print(f"Dataset created with {len(self.features)} features")
        print(f"Features: {self.features[:5]}...")  # Print first 5 features for verification

    def __len__(self):
        return len(self.df) - (self.max_length or 1)

    def __getitem__(self, idx):
        seq_start = idx
        seq_end = idx + self.max_length

        seq = self.df.iloc[seq_start:seq_end]
        x = torch.tensor(seq[self.features].values, dtype=torch.float32)
        y = torch.tensor(self.df[self.target].iloc[seq_end - 1], dtype=torch.float32)  # Fixed: predict current, not future

        return x, y

In [18]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=HIDDEN_SIZE, num_layers=N_LAYERS, 
                 bidirectional=BIDIRECTIONAL, dropout=DROP, n_clf_layers=N_CLF_LAYERS):
        super().__init__()
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        
        self.dropout = nn.Dropout(dropout)
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=False,
        )
        
        output_size = 2 * hidden_size if bidirectional else hidden_size
        
        # Regression head
        clf_layers = []
        for i in range(n_clf_layers - 1):
            clf_layers.append(nn.Linear(output_size, output_size))
            clf_layers.append(nn.ReLU())
            clf_layers.append(nn.Dropout(dropout))
        clf_layers.append(nn.Linear(output_size, 1))
        
        self.regressor = nn.Sequential(*clf_layers)
        self._reinitialize()
        
    def _reinitialize(self):
        for name, p in self.named_parameters():
            if 'lstm' in name:
                if 'weight_ih' in name:
                    nn.init.xavier_uniform_(p.data)
                elif 'weight_hh' in name:
                    nn.init.orthogonal_(p.data)
                elif 'bias_ih' in name:
                    p.data.fill_(0)
                    # Set forget-gate bias to 1
                    n = p.size(0)
                    p.data[(n // 4):(n // 2)].fill_(1)
                elif 'bias_hh' in name:
                    p.data.fill_(0)
            elif any(layer_name in name for layer_name in ['regressor', 'Linear']):
                if 'weight' in name:
                    nn.init.xavier_uniform_(p.data)
                elif 'bias' in name:
                    p.data.fill_(0)

    def forward(self, x, mask):
        # x shape: (seq_len, batch_size, input_size)
        # mask shape: (seq_len, batch_size)
        
        x, _ = self.lstm(x)
        
        # Get the last valid output for each sequence
        seq_lengths = mask.sum(dim=0)  # (batch_size,)
        batch_size = x.size(1)
        last_outputs = []
        
        for i in range(batch_size):
            last_idx = int(seq_lengths[i]) - 1
            last_idx = max(0, min(last_idx, x.size(0) - 1))  # Ensure valid index
            last_outputs.append(x[last_idx, i, :])
        
        x = torch.stack(last_outputs, dim=0)
        x = self.dropout(x)
        
        return self.regressor(x)

In [19]:
def collate_fn(batch):
    x_tensors = [item[0] for item in batch]
    y_tensors = [item[1] for item in batch]

    # Pad sequences
    batch_x = pad_sequence(x_tensors, batch_first=False, padding_value=0.0)
    batch_y = torch.stack(y_tensors)
    
    # Create mask
    mask = pad_sequence(
        [torch.ones(len(x)) for x in x_tensors], 
        batch_first=False, 
        padding_value=0
    )
    
    return batch_x, batch_y, mask


In [20]:
# Create datasets
ds_train = ZelestraData(train_scaled, max_length=max_length)
ds_test = ZelestraData(test_scaled, max_length=max_length)

Dataset created with 89 features
Features: ['meteorolgicas_em_03_02_gii', 'meteorolgicas_em_08_01_gii', 'meteorolgicas_em_03_02_ghi', 'meteorolgicas_em_08_01_ghi', 'meteorolgicas_em_08_01_gii_rear']...
Dataset created with 89 features
Features: ['meteorolgicas_em_03_02_gii', 'meteorolgicas_em_08_01_gii', 'meteorolgicas_em_03_02_ghi', 'meteorolgicas_em_08_01_ghi', 'meteorolgicas_em_08_01_gii_rear']...


In [21]:
train_loader = DataLoader(
    ds_train, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn, 
    drop_last=True
)

valid_loader = DataLoader(
    ds_test,
    shuffle=False,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    drop_last=True,
)

print(f"Training dataset contains: {len(ds_train)} sequences")
print(f"Validation dataset contains: {len(ds_test)} sequences")

Training dataset contains: 11656 sequences
Validation dataset contains: 5716 sequences


In [22]:
# Initialize model
model = LSTMRegressor(input_size=len(all_features)).to(DEVICE)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

# Add gradient clipping for stability
max_grad_norm = 1.0

In [23]:
def evaluate(model, data_loader, loss_fn):
    model.eval()
    total_loss = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for x, y, mask in data_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            mask = mask.to(DEVICE)
            
            # Check for NaN values
            if torch.isnan(x).any() or torch.isnan(y).any():
                print("NaN detected in input data!")
                continue
                
            logits = model(x, mask)
            loss = loss_fn(logits.squeeze(-1), y)
            
            if not torch.isnan(loss):
                total_loss += loss.item()
                num_batches += 1
    
    model.train()
    return total_loss / max(num_batches, 1)

In [24]:
class Monitor:
    def __init__(self):
        self.records = collections.defaultdict(list)

    def add(self, metric_name, epoch, value):
        self.records[metric_name].append({"epoch": epoch, "value": value})
        print(f"Epoch {epoch}/{EPOCHS} - {metric_name}: {value:.4f}")

    @property
    def dataframe(self):
        return pd.DataFrame(
            {
                k: pd.DataFrame(v).rename(columns={"value": k}).set_index("epoch")[k]
                for k, v in self.records.items()
            }
        )

In [25]:
# Training loop
print("=== STARTING TRAINING ===")
best_val_loss = float('inf')
monitor = Monitor()

for epoch in tqdm(range(1, EPOCHS + 1)):
    # Training
    model.train()
    losses = []
    
    for batch_idx, (x, y, mask) in enumerate(train_loader):
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        mask = mask.to(DEVICE)
        
        # Check for NaN values in input
        if torch.isnan(x).any() or torch.isnan(y).any():
            print(f"NaN detected in batch {batch_idx}!")
            continue
        
        optimizer.zero_grad()
        logits = model(x, mask)
        loss = loss_fn(logits.squeeze(-1), y)
        
        # Check for NaN loss
        if torch.isnan(loss):
            print(f"NaN loss detected in epoch {epoch}, batch {batch_idx}!")
            continue
        
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        
        optimizer.step()
        losses.append(loss.item())
        
        # Debug: Print loss for first few batches
        if epoch == 1 and batch_idx < 5:
            print(f"Batch {batch_idx}: loss = {loss.item():.4f}")
    
    if losses:  # Only calculate if we have valid losses
        train_loss = np.mean(losses)
        monitor.add("train_loss", epoch, train_loss)
        
        # Validation
        if epoch % 5 == 0:  # Validate every 5 epochs
            val_loss = evaluate(model, valid_loader, loss_fn)
            monitor.add("val_loss", epoch, val_loss)
            
            # Checkpointing
            if val_loss < best_val_loss:
                print(f"Validation loss improved from {best_val_loss:.4f} to {val_loss:.4f}")
                torch.save(model.state_dict(), "model_best.pt")
                best_val_loss = val_loss
    else:
        print(f"Epoch {epoch}: No valid losses computed!")

=== STARTING TRAINING ===


  0%|          | 0/50 [00:00<?, ?it/s]

Batch 0: loss = 0.5932
Batch 1: loss = 1.4653
Batch 2: loss = 0.5481
Batch 3: loss = 0.5917
Batch 4: loss = 0.3152
Epoch 1/50 - train_loss: 0.0493
Epoch 2/50 - train_loss: 0.0221
Epoch 3/50 - train_loss: 0.0194
Epoch 4/50 - train_loss: 0.0193
Epoch 5/50 - train_loss: 0.0185
Epoch 5/50 - val_loss: 0.0119
Validation loss improved from inf to 0.0119
Epoch 6/50 - train_loss: 0.0182
Epoch 7/50 - train_loss: 0.0166
Epoch 8/50 - train_loss: 0.0164
Epoch 9/50 - train_loss: 0.0166
Epoch 10/50 - train_loss: 0.0152
Epoch 10/50 - val_loss: 0.0085
Validation loss improved from 0.0119 to 0.0085
Epoch 11/50 - train_loss: 0.0142
Epoch 12/50 - train_loss: 0.0148
Epoch 13/50 - train_loss: 0.0136
Epoch 14/50 - train_loss: 0.0136
Epoch 15/50 - train_loss: 0.0134
Epoch 15/50 - val_loss: 0.0101
Epoch 16/50 - train_loss: 0.0142
Epoch 17/50 - train_loss: 0.0133
Epoch 18/50 - train_loss: 0.0148
Epoch 19/50 - train_loss: 0.0136
Epoch 20/50 - train_loss: 0.0139
Epoch 20/50 - val_loss: 0.0082
Validation loss impr

In [26]:
print("=== TRAINING COMPLETED ===")
print(f"Best validation loss: {best_val_loss:.4f}")

=== TRAINING COMPLETED ===
Best validation loss: 0.0077


In [27]:
(1-np.sqrt(best_val_loss))*100

91.20559439774699